In [0]:
#Regression model.
# =====================================
# Import Libraries
# =====================================

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator


In [0]:
#Load Feature Engineered Dataset
# =====================================
# Load Feature Engineered Dataset
# =====================================

df = spark.read.format("delta").load(
    "/Volumes/workspace/default/uber_data/feature_engineered_delta"
)

print("Dataset Loaded Successfully")
display(df.limit(5))

Dataset Loaded Successfully


In [0]:
#Select Features and Label
# =====================================
# Select Features and Label
# =====================================

ml_df = df.select("features", "fare_amount")

display(ml_df.limit(5))

In [0]:
#Train-Test Split
# =====================================
# Split Dataset
# =====================================

train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

print("Training Records :", train_df.count())
print("Testing Records :", test_df.count())

Training Records : 39943
Testing Records : 10054


In [0]:
# =====================================
# Train Linear Regression Model
# =====================================

lr = LinearRegression(
    featuresCol="features",
    labelCol="fare_amount"
)

model = lr.fit(train_df)

print("Model Trained Successfully")

Model Trained Successfully


In [0]:
train_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/uber_data/train_dataset"
)

print("Train Dataset Saved")

In [0]:
test_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/uber_data/test_dataset"
)

print("Test Dataset Saved")

Test Dataset Saved


In [0]:
from pyspark.sql.functions import abs, col

prediction_df = predictions.select(
    "fare_amount",
    "prediction"
)

prediction_df = prediction_df.withColumn(
    "prediction_error",
    abs(col("fare_amount") - col("prediction"))
)

display(prediction_df.limit(10))

fare_amount,prediction,prediction_error
2.38,9.797309561516983,7.417309561516983
3.56,13.242487765867065,9.682487765867064
1.61,5.930118625467433,4.320118625467432
3.72,10.06426403004542,6.344264030045419
1.08,4.416699258884817,3.336699258884817
1.74,5.3806523834939455,3.6406523834939453
1.96,5.668039251953932,3.708039251953932
2.8,6.968484772133102,4.168484772133102
3.21,7.48643377644254,4.27643377644254
2.12,5.308441343783956,3.188441343783956


In [0]:
prediction_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/uber_data/prediction_dataset"
)

print("Prediction Dataset Saved Successfully")

Prediction Dataset Saved Successfully


In [0]:
#Predictions
# =====================================
# Predict Fare
# =====================================

predictions = model.transform(test_df)

display(
    predictions.select(
        "fare_amount",
        "prediction"
    ).limit(10)
)

fare_amount,prediction
2.38,9.797309561516983
3.56,13.242487765867065
1.61,5.930118625467433
3.72,10.06426403004542
1.08,4.416699258884817
1.74,5.3806523834939455
1.96,5.668039251953932
2.8,6.968484772133102
3.21,7.48643377644254
2.12,5.308441343783956


In [0]:
predictions = model.transform(test_df)

display(
    predictions.select(
        "fare_amount",
        "prediction"
    ).limit(10)
)

fare_amount,prediction
2.38,9.797309561516983
3.56,13.242487765867065
1.61,5.930118625467433
3.72,10.06426403004542
1.08,4.416699258884817
1.74,5.3806523834939455
1.96,5.668039251953932
2.8,6.968484772133102
3.21,7.48643377644254
2.12,5.308441343783956


In [0]:
# =====================================
# Evaluate Model
# =====================================

rmse_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

r2_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

rmse = rmse_evaluator.evaluate(predictions)
r2 = r2_evaluator.evaluate(predictions)

print("RMSE :", rmse)
print("R2 Score :", r2)

RMSE : 3.0587143116631053
R2 Score : 0.7600905497618218


In [0]:
#Display Predictions
# =====================================
# Sample Predictions
# =====================================

display(
    predictions.select(
        "fare_amount",
        "prediction"
    )
)

fare_amount,prediction
2.38,9.797309561516983
3.56,13.242487765867065
1.61,5.930118625467433
3.72,10.06426403004542
1.08,4.416699258884817
1.74,5.3806523834939455
1.96,5.668039251953932
2.8,6.968484772133102
3.21,7.48643377644254
2.12,5.308441343783956


In [0]:
#Save Model
# =====================================
# Save Machine Learning Model
# =====================================

model.write().overwrite().save(
    "/Volumes/workspace/default/uber_data/uber_fare_prediction_model"
)

print("Model Saved Successfully")

Model Saved Successfully


In [0]:
print("=" * 50)
print("ETL PIPELINE COMPLETED")
print("=" * 50)

print("Raw Dataset           : Saved")
print("Clean Dataset         : Saved")
print("Feature Dataset       : Saved")
print("Prediction Dataset    : Saved")
print("Pipeline Status       : SUCCESS")

ETL PIPELINE COMPLETED
Raw Dataset           : Saved
Clean Dataset         : Saved
Feature Dataset       : Saved
Prediction Dataset    : Saved
Pipeline Status       : SUCCESS


In [0]:
# Read Prediction Dataset
prediction_df = spark.read.parquet(
    "/Volumes/workspace/default/uber_data/prediction_dataset"
)

# Save as CSV
prediction_df.coalesce(1).write \
.mode("overwrite") \
.option("header", "true") \
.csv("/Volumes/workspace/default/uber_data/prediction_csv")

print("Prediction CSV Saved Successfully")

Prediction CSV Saved Successfully
